In [1]:
!pip install osmnx sqlalchemy psycopg2-binary geoalchemy2 geopandas folium gradio transformers accelerate accelerate bitsandbytes librosa langdetect langchain langchain-community langchain-huggingface sentence-transformers faiss-cpu pypdf wikipedia ddgs

In [2]:
DOMAIN_HANDBOOKS = {
    "College Campus": """
UNIVERSAL COLLEGE CAMPUS RULES AND GUIDELINES:
1. Anti-Ragging Policy: Ragging is strictly prohibited under UGC regulations. Any form of ragging is a punishable offense and can lead to expulsion and criminal charges.
2. ID Cards: All students and staff must carry valid ID cards at all times on campus. Visitors must register at the main gate with valid government ID.
3. Hostel Curfew: Most hostels enforce a curfew between 10:00 PM and 6:00 AM. Late entry requires prior permission from the warden.
4. Library Rules: Silence must be maintained in the library. Food and drinks are not allowed. Books must be returned within the due date to avoid fines.
5. Speed Limit: The campus speed limit is typically 20-30 km/h. Reckless driving is strictly prohibited.
6. Substance Abuse: Smoking, alcohol, and drugs are strictly banned on campus premises.
7. Academic Integrity: Plagiarism and cheating are serious offenses. All submitted work must be original.
8. Dress Code: Formal attire is required in laboratories and during examinations. Casual clothing is acceptable in hostels and common areas.
9. Guest Policy: Overnight guests in hostels require prior approval from the hostel warden.
10. Emergency: Campus security can be reached 24/7. Fire assembly points are marked across campus.
11. Wi-Fi: Campus Wi-Fi is available in most academic buildings and libraries. Students must use their institutional credentials to log in.
12. Mess Timings: Breakfast (7:30-9:30 AM), Lunch (12:00-2:00 PM), Snacks (4:30-5:30 PM), Dinner (7:30-9:30 PM).
""",

    "Temple / Monument": """
UNIVERSAL TEMPLE AND MONUMENT VISITOR GUIDELINES:
1. Dress Code: Visitors must wear modest clothing. Shoulders and knees should be covered. Sleeveless tops, shorts, and mini-skirts are generally not allowed inside temple sanctums.
2. Footwear: Shoes and sandals must be removed before entering the temple premises. Shoe storage is usually available near the entrance.
3. Photography: Photography is typically prohibited inside the main sanctum or shrine. Outdoor photography is usually allowed. Flash photography may damage ancient artwork and is banned.
4. Silence: Visitors should maintain silence and decorum inside the temple. Loud conversations and phone calls are discouraged.
5. Offerings: Devotees may bring flowers, fruits, and coconuts as offerings. Verify with local customs as specific temples may have restrictions.
6. Timings: Most temples are open from 6:00 AM to 12:00 PM and 4:00 PM to 9:00 PM. Some temples close during the afternoon.
7. Entry Fee: Many ASI (Archaeological Survey of India) monuments charge approximately Rs 40 for Indian citizens and Rs 600 for foreign nationals. Children under 15 are usually free.
8. Guides: Official guides are available at the entrance of major monuments. Always verify their ID before hiring.
9. Food & Drink: Outside food and beverages are generally not allowed inside monument premises. Water bottles are usually permitted.
10. Touching Artifacts: Visitors must not touch, lean on, or climb ancient structures, statues, or carvings.
11. Littering: Littering is strictly prohibited. Use designated dustbins.
12. Pet Policy: Pets are generally not allowed inside temples and monuments.
""",

    "Tourist Area": """
UNIVERSAL TOURIST AREA GUIDELINES:
1. Entry Tickets: Most tourist attractions have separate pricing for Indian nationals and foreign tourists. Online booking is recommended to avoid long queues.
2. Guides: Licensed guides are available at major tourist spots. Always verify their official ID. Negotiate rates before starting the tour.
3. Safety: Keep your belongings secure. Beware of pickpockets in crowded areas. Keep copies of important documents.
4. Photography: Check if photography is allowed and whether there is an additional camera fee. Drone photography requires special permission.
5. Local Transport: Auto-rickshaws and taxis are common. Always agree on the fare before starting or insist on using the meter. Ride-sharing apps are available in most cities.
6. Food Safety: Drink bottled water. Be cautious with street food if you have a sensitive stomach. Prefer busy stalls with high turnover.
7. Weather: Check weather conditions before visiting. Carry sunscreen, hats, and water during summer. Carry rain gear during monsoon season (June-September).
8. Timings: Most tourist places are open from 9:00 AM to 5:00 PM. They are usually closed on Mondays or specific national holidays.
9. Bargaining: Bargaining is common and expected at local markets and souvenir shops. Start at about 50% of the quoted price.
10. Emergency Numbers: Police: 100, Ambulance: 108, Fire: 101, Tourist Helpline: 1363.
11. Respect Local Culture: Be respectful of local customs, traditions, and religious practices. Ask permission before photographing local people.
12. Environment: Do not litter. Carry a reusable water bottle. Avoid single-use plastics at nature reserves and beaches.
"""
}

In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

rag_vector_db = None
# NEW: Forcing the embeddings model onto the Mac GPU ("mps")
rag_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cuda'}
)

def process_pdf(filepath):
    global rag_vector_db
    if filepath is None: return "No file uploaded."
    try:
        loader = PyPDFLoader(filepath)
        docs = loader.load()
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=200)
        splits = text_splitter.split_documents(docs)
        pdf_db = FAISS.from_documents(splits, rag_embeddings)

        # MERGE into existing DB instead of replacing
        if rag_vector_db is not None:
            rag_vector_db.merge_from(pdf_db)
            return f"✅ PDF processed! Merged {len(splits)} chunks into existing RAG memory."
        else:
            rag_vector_db = pdf_db
            return f"✅ PDF processed! Loaded {len(splits)} chunks into RAG memory."
    except Exception as e:
        return f"❌ Error processing PDF: {str(e)}"



/var/folders/tb/rlnnqcns7h17x7h9zrrcbbtr0000gn/T/ipykernel_42208/3680092341.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [5]:
# --- 2. LOAD MODELS ---
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig  # <--- Apple's MLX library!

print("🚀 Loading SeamlessM4T (ASR + Translation)...")
asr_processor = AutoProcessor.from_pretrained("facebook/seamless-m4t-v2-large")
asr_model = SeamlessM4Tv2Model.from_pretrained("facebook/seamless-m4t-v2-large", torch_dtype=torch.float16).to("cuda")

print("🚀 Loading Qwen2.5-Coder (SQL + Chat) in 4-bit via MLX...")
# Use the pre-quantized 4-bit Mac version!
sql_model_name = "Qwen/Qwen2.5-Coder-7B-Instruct"

# mlx_lm handles the quantization automatically on Apple Silicon
    bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    sql_model = AutoModelForCausalLM.from_pretrained(sql_model_name, device_map="auto", quantization_config=bnb_config)

print("✅ Both models loaded successfully into Mac Unified Memory!")

🚀 Loading SeamlessM4T (ASR + Translation)...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

[transformers] Instantiating a decoder SeamlessM4Tv2Attention without passing `layer_idx` is not recommended and will lead to errors during the forward call, if caching is used. Please make sure to provide a `layer_idx` when creating this class.


Loading weights:   0%|          | 0/2232 [00:00<?, ?it/s]

🚀 Loading Qwen2.5-Coder (SQL + Chat) in 4-bit via MLX...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

✅ Both models loaded successfully into Mac Unified Memory!


In [9]:
# --- 3. DATA LOADING LOGIC ---
def load_map_data(location_name):
    from sqlalchemy import text # <--- ADD THIS LINE HERE!
    global CURRENT_LOCATION_NAME, CURRENT_LOCATION_LAT, CURRENT_LOCATION_LON, rag_vector_db


    if not location_name.strip(): return "❌ Please enter a valid location name."
    print(f"\n🌍 Geocoding '{location_name}'...")
    try:
        lat, lon = ox.geocode(location_name)
        CURRENT_LOCATION_LAT = lat
        CURRENT_LOCATION_LON = lon

    except Exception as e:
        return f"❌ Could not find location for '{location_name}'. Try adding the city name."
    print(f"🌍 Fetching spatial data for {location_name} at ({lat}, {lon})...")
    try:
        # MASSIVELY EXPANDED TAGS FOR TOURISM & HERITAGE
        tags = {
            'amenity': ['place_of_worship', 'toilets', 'parking', 'drinking_water', 'food_court', 'cafe', 'restaurant', 'library', 'hospital', 'clinic', 'college', 'university', 'bank', 'atm'],
            'tourism': ['museum', 'attraction', 'information', 'viewpoint', 'hotel', 'hostel'],
            'historic': ['monument', 'ruins', 'fort'],
            'religion': True,
            'shop': ['souvenir', 'ticket', 'convenience', 'supermarket', 'mall'],
            'building': ['dormitory', 'university', 'temple', 'church', 'mosque', 'train_station', 'hospital'],
            'leisure': ['park', 'pitch', 'sports_centre', 'garden'],
            'man_made': True, 'natural': True, 'highway': ['bus_stop'],
            'wheelchair': True, 'opening_hours': True, 'cuisine': True,
            'diet:vegan': True, 'takeaway': True, 'internet_access': True, 'dog': True,
            'sport': True, 'office': True, 'healthcare': True
        }

        gdf = ox.features_from_point((lat, lon), tags=tags, dist=2000).reset_index()
        keep_cols = [c for c in gdf.columns if c != 'geometry'] + ['geometry']
        gdf = gdf[keep_cols]

        for col in gdf.select_dtypes(include=['object', 'category']).columns:
            if col != 'geometry':
                gdf[col] = gdf[col].astype(object).where(gdf[col].notna(), None)

        gdf = gdf.set_crs(epsg=4326) if gdf.crs is None else gdf.to_crs(epsg=4326)

        with ENGINE.connect() as conn:
            conn.execute(text("CREATE EXTENSION IF NOT EXISTS postgis;"))
            conn.commit()

        print(f"💾 Writing {len(gdf)} features to PostgreSQL (campus_places)...")
        gdf.to_postgis("campus_places", ENGINE, if_exists='replace', index=False)

        CURRENT_LOCATION_NAME = location_name

        # --- AUTO-FETCH WIKIPEDIA FOR RAG ---
        try:
            # Tell Wikipedia we are a real app so they don't block us!
            wikipedia.set_user_agent("UniversalAIGuide/1.0 (student@college.edu)")

            # Simplify the search term for Wikipedia
            wiki_search = wikipedia.search(location_name, results=1)
            if wiki_search:
                wiki_page = wikipedia.page(wiki_search[0], auto_suggest=False)
                wiki_text = wiki_page.content

                # Correct LangChain imports
                from langchain_core.documents import Document
                from langchain_text_splitters import RecursiveCharacterTextSplitter

                # THE FAISS CROWDING FIX IS HERE (chunk_size=400)
                text_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=100)
                wiki_docs = [Document(page_content=chunk) for chunk in text_splitter.split_text(wiki_text)]
                wiki_db = FAISS.from_documents(wiki_docs, rag_embeddings)

                                # Wipe the old database clean when loading a new map to prevent duplicates!
                rag_vector_db = wiki_db

                print(f"📚 Wikipedia loaded! {len(wiki_docs)} chunks from '{wiki_page.title}' added to RAG.")
            else:
                print(f"⚠️ No Wikipedia article found for '{location_name}'.")
        except Exception as e:
            print(f"⚠️ Wikipedia fetch failed (will rely on handbooks): {e}")

                # --- AUTO-FETCH DIRECT FAQ URL ---
        global DIRECT_FAQ_URL
        DIRECT_FAQ_URL = ""
        try:
            from ddgs import DDGS
            search_query = f"{location_name} FAQ -site:wikipedia.org"
            results = list(DDGS().text(search_query, max_results=3))
            
            if results:
                # Look for the exact FAQ page
                for res in results:
                    url = res.get('href', '').lower()
                    title = res.get('title', '').lower()
                    if 'faq' in url or 'faq' in title or 'frequently' in title:
                        DIRECT_FAQ_URL = res.get('href')
                        break
                
                # Fallback to the first non-Wikipedia result
                if not DIRECT_FAQ_URL:
                    for res in results:
                        fallback_url = res.get('href', '')
                        if 'wikipedia.org' not in fallback_url:
                            DIRECT_FAQ_URL = fallback_url
                            break
                    
                print(f"✅ Found Direct FAQ Link: {DIRECT_FAQ_URL}")
        except Exception as e:
            print(f"⚠️ Could not fetch FAQ link: {e}")


        # NOW WE RETURN SUCCESS!
        return f"✅ Map data for '{location_name}' successfully loaded! ({len(gdf)} locations found)"
    
    except Exception as e:
        return f"❌ Failed to build map data: {str(e)}"


def ui_audit_data():
    try:
        with ENGINE.connect() as conn:
            total = conn.execute(text("SELECT COUNT(*) FROM campus_places")).scalar()
            named = conn.execute(text("SELECT COUNT(*) FROM campus_places WHERE name IS NOT NULL AND name != 'nan'")).scalar()
        return f"✅ Audit complete! Database currently holds {total} places ({named} named) for {CURRENT_LOCATION_NAME}."
    except Exception:
        return "❌ Audit failed. Ensure you have loaded a map first."


def get_db_results(sql):
    if not sql: return None
    if "select " in sql.lower():
        select_clause = sql.lower().split("from")[0]
        if "geometry" not in select_clause:
            if "join " in sql.lower():
                match = re.search(r'(?i)select\s+(distinct\s+)?([a-z0-9_]+)\.', sql)
                if match:
                    alias = match.group(2)
                    sql = re.sub(r'(?i)select\s+(distinct\s+)?', f'SELECT \\1{alias}.geometry, ', sql, count=1)
                else:
                    sql = re.sub(r'(?i)select\s+(distinct\s+)?', r'SELECT \1geometry, ', sql, count=1)
            else:
                sql = re.sub(r'(?i)select\s+(distinct\s+)?', r'SELECT \1geometry, ', sql, count=1)
    try:
        safe_sql = sql.replace("%", "%%")
        return gpd.read_postgis(safe_sql, ENGINE, geom_col='geometry')
    except Exception as e:
        print(f"💻 [DB Error]: {e}")
        return None


def format_results_for_llm(gdf):
    if gdf is None or len(gdf) == 0: return "No specific places found for this query."
    results = gdf.drop(columns=['geometry']).to_dict(orient='records')
    lines = []
    for r in results:
        parts = {k: v for k, v in r.items() if pd.notnull(v) and str(v).lower() != 'nan'}

        # 1. Try to get the real name
        name = parts.pop('name', None)

        # 2. Intelligent Fallback if name is missing/nan
        if not name or str(name).lower() == 'nan' or str(name).lower() == 'none':
            for tag in ['amenity', 'tourism', 'historic', 'shop', 'leisure', 'building']:
                if tag in parts:
                    name = str(parts[tag]).replace("_", " ").title()
                    break
            if not name:
                name = 'Unknown Location'

        detail = ", ".join(f"{k}: {v}" for k, v in parts.items() if v)
        lines.append(f"- {name}" + (f" ({detail})" if detail else ""))
    return "\n".join(lines)


def generate_map_html(gdf):
    if gdf is None or len(gdf) == 0:
        return "<div style='padding:20px;text-align:center;color:#666;'><h3>❌ No map data found for this query.</h3></div>"

    gdf = gdf.set_crs(epsg=4326) if gdf.crs is None else gdf.to_crs(epsg=4326)

    if 'ref_geom' in gdf.columns:
        def parse_wkb(x):
            if pd.isna(x): return None
            try: return shapely.wkb.loads(str(x), hex=True)
            except: return None
        gdf['ref_geom'] = gdf['ref_geom'].apply(parse_wkb)

    try:
        centroid = gdf.geometry.centroid
        m = folium.Map(location=[centroid.y.mean(), centroid.x.mean()], zoom_start=16, tiles="CartoDB positron")
        marker_cluster = MarkerCluster().add_to(m)
        drawn_refs = set()

        # --- NEW: ALWAYS ADD THE MAIN LOCATION HUB MARKER ---
        global CURRENT_LOCATION_NAME, CURRENT_LOCATION_LAT, CURRENT_LOCATION_LON
        if 'CURRENT_LOCATION_NAME' in globals() and 'CURRENT_LOCATION_LAT' in globals() and 'CURRENT_LOCATION_LON' in globals():
            folium.Marker(
                [CURRENT_LOCATION_LAT, CURRENT_LOCATION_LON],
                popup=folium.Popup(f"<div style='min-width: 150px;'><h4>{CURRENT_LOCATION_NAME} (Main Hub)</h4></div>", max_width=300),
                icon=folium.Icon(color="green", icon="home")
            ).add_to(m)
        # ----------------------------------------------------

        for idx, row in gdf.iterrows():
            name = str(row.get('name', ''))
            if not name or name.lower() == 'nan' or name.lower() == 'none':
                for tag in ['amenity', 'tourism', 'historic', 'shop', 'leisure', 'building']:
                    val = row.get(tag)
                    if pd.notnull(val) and str(val).lower() != 'nan':
                        name = str(val).replace("_", " ").title()
                        break
                if not name:
                    name = 'Unknown Location'

            if row.geometry is not None and not row.geometry.is_empty:
                lat, lon = row.geometry.centroid.y, row.geometry.centroid.x
                details = "".join([f"<b>{col}:</b> {val}<br>" for col, val in row.items() if col not in ['name', 'geometry', 'ref_geom', 'ref_name'] and pd.notnull(val) and str(val).lower() != 'nan'])
                popup_html = f"<div style='min-width: 150px;'><h4>🎯 {name}</h4>{details}</div>"
                folium.Marker([lat, lon], popup=folium.Popup(popup_html, max_width=300), icon=folium.Icon(color="red", icon="info-sign")).add_to(marker_cluster)

            if 'ref_geom' in gdf.columns and 'ref_name' in gdf.columns:
                ref_geom, ref_name = row['ref_geom'], str(row['ref_name'])
                if ref_geom is not None and not ref_geom.is_empty:
                    ref_lat, ref_lon = ref_geom.centroid.y, ref_geom.centroid.x
                    ref_key = f"{ref_name}_{ref_lat}_{ref_lon}"
                    if ref_key not in drawn_refs:
                        drawn_refs.add(ref_key)

                        if not ref_name or ref_name.lower() == 'nan' or ref_name.lower() == 'none':
                            ref_name = 'Reference Location'

                        folium.Marker([ref_lat, ref_lon], popup=folium.Popup(f"<div style='min-width: 150px;'><h4>📍 {ref_name} (Reference)</h4></div>", max_width=300), icon=folium.Icon(color="blue", icon="star")).add_to(m)

        map_rendered = m.get_root().render().replace('"', '&quot;')
        return f'<iframe srcdoc="{map_rendered}" style="width: 100%; height: 500px; border: none; border-radius: 12px; box-shadow: 0 4px 12px rgba(0,0,0,0.1);"></iframe>'
    except Exception as e:
        print(f"Map generation failed: {e}")
        return "<div style='padding:20px;text-align:center;'><h3>❌ Map Generation Failed.</h3></div>"

In [10]:
# --- 4. DYNAMIC PROMPT TEMPLATE (Model-Agnostic) ---
system_prompt_template = """You are an expert PostgreSQL + PostGIS developer and a Local Expert Guide for {location_name}.
Generate a VALID PostGIS SQL query for the user's question.

The database is OpenStreetMap-style and highly denormalized.
You MUST understand semantic tag categories before writing queries.

====================================================================
DATABASE SCHEMA: Table → campus_places
====================================================================

-------------------------
1️⃣ CORE IDENTIFIERS
-------------------------
- element (node, way, relation)
- id (bigint OSM ID)
- name
- short_name
- official_name
- alt_name
- name:en
- name:hi

-------------------------
2️⃣ GEOSPATIAL
-------------------------
- geometry (PostGIS geometry column)

-------------------------
3️⃣ PRIMARY PLACE CATEGORY TAGS
-------------------------

amenity (UNIQUE VALUES):
- restaurant, blood_bank, cafe, club, clinic, research_institute
- hospital, conference_centre, ice_cream, fuel, fast_food
- college, dentist, school, place_of_worship, library
- post_office, courthouse, food_court, pharmacy, university
- bank, atm, toilets, parking, drinking_water

building (UNIQUE VALUES):
- university=department, dormitory, apartments, residential
- yes, hospital, university, temple, church, mosque, train_station

office (UNIQUE VALUES):
- yes, government, advertising_agency

shop (UNIQUE VALUES):
- hairdresser, supermarket, department_store, convenience, bakery
- souvenir, ticket, mall

healthcare (UNIQUE VALUES):
- hospital, pharmacy, clinic

tourism (UNIQUE VALUES):
- artwork, gallery, museum, hotel, viewpoint, hostel, attraction

leisure (UNIQUE VALUES):
- nature_reserve, pitch, park, garden
- sports_centre, swimming_pool, playground

sport (UNIQUE VALUES):
- badminton, cricket, table_tennis, multi, volleyball, skateboard

historic (UNIQUE VALUES):
- manor, aircraft, memorial, monument, ruins, fort

religion (UNIQUE VALUES):
- hindu, muslim, christian, buddhist

education (UNIQUE VALUES):
- school

man_made (UNIQUE VALUES):
- silo

artwork_type (UNIQUE VALUES):
- statue

(NEW ADVANCED TAGS):
- wheelchair (e.g., 'yes')
- opening_hours (e.g., '24/7')
- cuisine (e.g., 'indian', 'chinese')
- diet:vegan (e.g., 'yes')
- takeaway (e.g., 'yes')
- internet_access (e.g., 'wlan')
- dog (e.g., 'yes')

====================================================================
🧠 LOCAL GUIDE INTENT MAPPING LOGIC (COMPREHENSIVE)
====================================================================
Map the user's emotional or contextual intent to the correct pragmatic tags:

1. ATMOSPHERICS / VIBE (Emotional Intent)
- Quiet/Relaxing → amenity ILIKE '%library%' OR leisure ILIKE '%park%' OR natural ILIKE '%water%' OR building ILIKE '%temple%'
- Lively/Energetic → amenity ILIKE '%bar%' OR amenity ILIKE '%pub%' OR leisure ILIKE '%stadium%'
- Aesthetic/Tourism → tourism IS NOT NULL OR historic IS NOT NULL OR artwork_type IS NOT NULL

2. SITUATIONAL / CONTEXTUAL (Who & Why)
- Family/Kid-Friendly → leisure ILIKE '%playground%' OR tourism ILIKE '%theme_park%'
- Pet-Friendly → "dog" = 'yes' OR leisure ILIKE '%park%'
- Group/Gathering → amenity ILIKE '%food_court%' OR amenity ILIKE '%restaurant%'
- Tourist Needs → shop ILIKE '%souvenir%' OR shop ILIKE '%ticket%' OR tourism ILIKE '%hotel%'

3. FRICTIONAL (Convenience & Effort)
- Fast/Quick → amenity ILIKE '%fast_food%' OR "takeaway" = 'yes'
- Accessible → "wheelchair" = 'yes'
- Budget/Cheap → amenity ILIKE '%canteen%' OR amenity ILIKE '%food_court%'

4. PRAGMATIC / FUNCTIONAL (Utility)
- Productivity/Study → "internet_access" = 'wlan' OR amenity ILIKE '%library%' OR building ILIKE '%university%'
- Utility/Errand → amenity ILIKE '%atm%' OR amenity ILIKE '%bank%' OR amenity ILIKE '%post_office%'
- Dietary (Vegan) → "diet:vegan" = 'yes'
- Late Night → "opening_hours" ILIKE '%24/7%'

5. 🚨 CRITICAL BUG FIX (THE LIBRARY RULE) 🚨
- If the user asks for a LIBRARY, you MUST use: amenity ILIKE '%library%'
- You are STRICTLY FORBIDDEN from using the 'education' column for libraries. Using 'education' for a library is a FATAL ERROR.

6. 🚨 CRITICAL BUG FIX (THE MISSING ANCHOR RULE) 🚨
- If the user asks for places using terms like "nearby", "here", "closest", or "nearest" BUT DOES NOT specify a reference building, you MUST just return a general list of places.
- You are STRICTLY FORBIDDEN from using any spatial joins (No ST_DWithin, No ST_Distance, No 'ON 1=1').
- Using ST_Distance or ST_DWithin without a concrete reference point is a FATAL ERROR.

====================================================================
🚨 CRITICAL SQL RULES
====================================================================
1. Use ILIKE '%term%' for ALL text matches.
2. Columns with ":" MUST use double quotes → "diet:vegan"
3. SELECT ONLY: name AND the primary filtered column(s)
4. ORDER BY name ASC
5. LIMIT 5
6. LOGIC GROUPS: If searching for multiple values in the SAME column (e.g. cafe or restaurant), use OR. If combining DIFFERENT columns (e.g. a cafe that is ALSO vegan), use AND.
7. Output ONLY valid PostgreSQL SQL. No explanation.
8. 🚨 MISSING ANCHOR 🚨: If the user asks for a "nearby", "here", "closest", or "nearest" place without a reference building, you are STRICTLY FORBIDDEN from doing a spatial join. Just return a standard SELECT list.
9. 🚨 COLUMN RULES 🚨 Do NOT search for hotels in `amenity`. Hotels are in `tourism`. Do NOT search for monuments in `amenity`. Monuments are in `historic`.

====================================================================
🗺️ SPATIAL EXPLICIT JOINS & DISTANCE RULES (CRITICAL)
====================================================================
If the user asks about physical distance ("near", "within X meters", "closest to") AND they explicitly name a reference location (e.g. "closest to the library"):
1. You MUST do a direct self-join: FROM campus_places target JOIN campus_places ref ON ... (NEVER use a subquery inside ST_DWithin! You are STRICTLY FORBIDDEN from using WITH clauses or CTEs!)
2. You MUST cast geometries to geography to measure in real-world meters: `::geography`
3. For "within X meters": Use `ST_DWithin(target.geometry::geography, ref.geometry::geography, X)`
4. For "nearest/closest" (ONLY when a reference location is provided): Join on `1=1` and use `ORDER BY ST_Distance(target.geometry::geography, ref.geometry::geography) ASC LIMIT 1`
5. You MUST SELECT both the target geometry AND the reference geometry, like this:
   `SELECT target.geometry, target.name, ref.geometry AS ref_geom, ref.name AS ref_name`

====================================================================
FEW-SHOT EXAMPLES
====================================================================

Tourist says: "Where is the nearest shoe-keeping stand to the main temple?"
SQL Query:
SELECT target.geometry, target.name, target.amenity, ref.geometry AS ref_geom, ref.name AS ref_name
FROM campus_places target
JOIN campus_places ref ON ref.building ILIKE '%temple%' OR ref.name ILIKE '%temple%'
WHERE (target.shop ILIKE '%ticket%' OR target.name ILIKE '%shoe%')
ORDER BY ST_Distance(target.geometry::geography, ref.geometry::geography) ASC
LIMIT 1;

Student says: "I need a quiet place to read"
SQL Query:
SELECT name, amenity, leisure FROM campus_places WHERE (amenity ILIKE '%library%' OR leisure ILIKE '%park%') ORDER BY name ASC LIMIT 5;

Student says: "Find a cafe within 500 meters of the library"
SQL Query:
SELECT target.geometry, target.name, target.amenity, ref.geometry AS ref_geom, ref.name AS ref_name
FROM campus_places target
JOIN campus_places ref ON ST_DWithin(target.geometry::geography, ref.geometry::geography, 500)
WHERE target.amenity ILIKE '%cafe%'
  AND ref.amenity ILIKE '%library%'
ORDER BY target.name ASC LIMIT 5;

Tourist says: "Where is the closest ATM to the hotel?"
SQL Query:
SELECT target.geometry, target.name, target.amenity, ref.geometry AS ref_geom, ref.name AS ref_name
FROM campus_places target
JOIN campus_places ref ON 1=1
WHERE target.amenity ILIKE '%atm%'
  AND (ref.tourism ILIKE '%hotel%' OR ref.name ILIKE '%hotel%')
ORDER BY ST_Distance(target.geometry::geography, ref.geometry::geography) ASC
LIMIT 1;

Tourist says: "Find vegan food near the hospital."
SQL Query:
SELECT target.geometry, target.name, target.amenity, target."diet:vegan", ref.geometry AS ref_geom, ref.name AS ref_name
FROM campus_places target
JOIN campus_places ref ON ST_DWithin(target.geometry::geography, ref.geometry::geography, 1000)
WHERE (target.amenity ILIKE '%restaurant%' OR target.amenity ILIKE '%cafe%') AND target."diet:vegan" = 'yes'
  AND (ref.amenity ILIKE '%hospital%' OR ref.building ILIKE '%hospital%')
ORDER BY target.name ASC LIMIT 5;

Tourist says: "Where is the nearest toilet around here?"
SQL Query:
```sql
SELECT name, amenity FROM campus_places WHERE amenity ILIKE '%toilets%' LIMIT 5;

User says: "Find the nearest cafe."
SQL Query:
```sql
SELECT name, amenity FROM campus_places WHERE amenity ILIKE '%cafe%' ORDER BY name ASC LIMIT 5;

"""

In [ ]:
# --- 5. AI GENERATION & AUDIO LOGIC ---

# Map standard 2-letter langdetect codes to SeamlessM4T 3-letter codes
LANG_MAP = {
    'hi': 'hin', 'bn': 'ben', 'mr': 'mar',
    'te': 'tel', 'ta': 'tam', 'pa': 'pan', 'en': 'eng'
}

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def _to_device(inputs):
    if DEVICE == "cuda":
        return inputs.to(DEVICE)
    return inputs.to(DEVICE)

def translate_text_offline(text_input):
    try:
        detected_lang = detect(text_input)
        src_lang = LANG_MAP.get(detected_lang, 'eng')
        if src_lang == 'eng':
            return text_input

        inputs = asr_processor(text=text_input, src_lang=src_lang, return_tensors="pt")
        inputs = _to_device(inputs)

        with torch.no_grad():
            output_tokens = asr_model.generate(
                **inputs,
                tgt_lang="eng",
                generate_speech=False
            )[0].cpu()

        return asr_processor.decode(output_tokens, skip_special_tokens=True)[0].strip()

    except Exception as e:
        print(f"Translation Error: {e}")
        return text_input


def process_audio_input(audio_path):
    if not audio_path:
        return ""

    try:
        audio_array, sr = librosa.load(audio_path, sr=16000)

        inputs = asr_processor(
            audio=audio_array,
            sampling_rate=sr,
            return_tensors="pt"
        )

        if DEVICE == "cuda":
            inputs = inputs.to(DEVICE, torch.float16)
        else:
            inputs = inputs.to(DEVICE)

        with torch.no_grad():
            predicted_ids = asr_model.generate(
                **inputs,
                tgt_lang="eng",
                generate_speech=False
            )[0].cpu()

        return asr_processor.decode(predicted_ids, skip_special_tokens=True)[0].strip()

    except Exception as e:
        print(f"ASR Error: {e}")
        return ""


def generate_audio_response(text, tgt_lang_code='eng'):
    try:
        clean_text = text.replace("*", "").replace("#", "").split("(Debug")[0].strip()
        if not clean_text:
            return None

        inputs = asr_processor(
            text=clean_text,
            src_lang=tgt_lang_code,
            return_tensors="pt"
        )
        inputs = _to_device(inputs)

        with torch.no_grad():
            audio_array = asr_model.generate(
                **inputs,
                tgt_lang=tgt_lang_code
            )[0].cpu().numpy().squeeze()

        gc.collect()

        sample_rate = asr_model.config.sampling_rate
        return (sample_rate, audio_array)

    except Exception as e:
        print(f"🔊 [Audio Error]: {e}")
        return None


def translate_english_to_target(english_text, tgt_lang_code):
    if tgt_lang_code == 'eng':
        return english_text

    try:
        inputs = asr_processor(
            text=english_text,
            src_lang='eng',
            return_tensors="pt"
        )
        inputs = _to_device(inputs)

        with torch.no_grad():
            output_tokens = asr_model.generate(
                **inputs,
                tgt_lang=tgt_lang_code,
                generate_speech=False
            )[0].cpu()

        translated_text = asr_processor.decode(
            output_tokens,
            skip_special_tokens=True
        )[0].strip()

        gc.collect()
        return translated_text

    except Exception as e:
        print(f"Reverse Translation Error: {e}")
        return english_text


def clean_sql(text):
    if "```sql" in text:
        return text.split("```sql")[1].split("```")[0].strip()
    return text.strip()


def generate(model, tokenizer, prompt, max_tokens=200, verbose=False):
    try:
        inputs = tokenizer([prompt], return_tensors="pt")
        inputs = _to_device(inputs)

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            pad_token_id=tokenizer.eos_token_id,
        )

        generated_text = tokenizer.decode(
            outputs[0][inputs.input_ids.shape[1]:],
            skip_special_tokens=True,
        )

        return generated_text.replace("<|im_end|>", "").strip()

    except Exception as e:
        print(f"[generate] error: {e}")
        return ""


def get_sql(question, feedback_context=""):
    system_content = system_prompt_template.format(location_name=CURRENT_LOCATION_NAME)
    if feedback_context:
        system_content += f"\n\n[SYSTEM FEEDBACK]:\n{feedback_context}\nFix the SQL query."

    user_content = f"Junior says: \"{question}\"\\n\\nSQL Query:"

    messages = [
        {"role": "system", "content": system_content},
        {"role": "user", "content": user_content}
    ]

    prompt = sql_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    raw_sql = generate(sql_model, sql_tokenizer, prompt=prompt, max_tokens=300)
    return clean_sql(raw_sql)


def get_friendly_response(original_question, db_results):
    if hasattr(db_results, 'drop'):
        result_str = format_results_for_llm(db_results)
    else:
        result_str = str(db_results)

    messages = [
        {
            "role": "system",
            "content": (
                f"You are a friendly Local Guide for {CURRENT_LOCATION_NAME}. "
                "Summarize these database results in a conversational way. "
                "CRITICAL: You MUST explicitly list the actual names of the places found in the database. "
                "CRITICAL: Keep your answer direct and use simple plain English. "
                "Do not use poetic or descriptive language. "
                "CRITICAL: You MUST reply entirely in English, no matter what language the user used. "
                "CRITICAL: Do NOT read, mention, or print any raw geometry coordinates or hex strings from the data."
                f"\\n\\nDatabase Results:\\n{result_str}"
            )
        },
        {"role": "user", "content": original_question}
    ]

    prompt = sql_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    return generate(sql_model, sql_tokenizer, prompt=prompt, max_tokens=150)


def generate_rag_response(question, target_lang_code):
    if rag_vector_db is None:
        error_msg = "Please upload a handbook or guidebook PDF first so I can answer policy questions."
        return translate_english_to_target(error_msg, target_lang_code)

    docs = rag_vector_db.similarity_search(question, k=15)
    unique_chunks = list(set([doc.page_content for doc in docs]))
    context = "\n\n".join(unique_chunks)

    prompt_text = (
        f"Answer the user's question using ONLY the provided Context. "
        f"If the context does not contain the answer, you MUST output exactly this exact phrase: NOT_FOUND\n\n"
        f"Context:\n{context}\n\nQuestion: {question}"
    )

    messages = [
        {
            "role": "system",
            "content": "You are a highly accurate RAG assistant. Your only job is to extract answers from the provided context."
        },
        {"role": "user", "content": prompt_text}
    ]

    prompt = sql_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    english_answer = generate(sql_model, sql_tokenizer, prompt=prompt, max_tokens=250)
    return translate_english_to_target(english_answer, target_lang_code)


def route_question(question):
    messages = [
        {
            "role": "system",
            "content": """You are an intent classifier. Output DB, POLICY, CHAT, or CLARIFICATION.
            RULES:
            - DB: Asking for places, food, amenities, or spatial locations with a reference.
            - POLICY: Asking about rules, guidelines, or history.
            - CHAT: Pure small talk.
            - CLARIFICATION: Asking for 'nearest', 'closest', or 'nearby' places BUT NO reference building is named.

            EXAMPLES:
            'Where is the library?' -> DB
            'Find the nearest cafe.' -> CLARIFICATION
            'Find the nearest cafe to the hostel.' -> DB
            'Where is the closest hospital?' -> CLARIFICATION
            'Are there ATMs nearby?' -> CLARIFICATION
            'Find a cafe within 500 meters of the library.' -> DB
            'Tell me a joke.' -> CHAT
            """
        },
        {"role": "user", "content": question}
    ]

    prompt = sql_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    return generate(sql_model, sql_tokenizer, prompt=prompt, max_tokens=10).strip().upper()


def get_chat_response(original_question, system_context):
    messages = [
        {"role": "system", "content": system_context},
        {"role": "user", "content": original_question}
    ]

    prompt = sql_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    return generate(sql_model, sql_tokenizer, prompt=prompt, max_tokens=200)


# --- 6. GRADIO WRAPPER & UI ---
def chat_wrapper(user_input, audio_input, history, tts_lang_choice, domain_choice):
    default_map = "<div style='height: 500px; display: flex; align-items: center; justify-content: center; background: #f8f9fa; border-radius: 12px; border: 1px solid #e0e0e0; color: #888;'><h4>Map will update after a spatial query</h4></div>"

    try:
        detected = detect(user_input) if user_input.strip() else 'en'
    except:
        detected = 'en'
    input_lang_code = LANG_MAP.get(detected, 'eng')

    audio_lang_code = tts_lang_choice.split("(")[1].replace(")", "")

    if audio_input:
        transcribed_text = process_audio_input(audio_input)
        if transcribed_text:
            user_input = transcribed_text
            english_query = transcribed_text
            input_lang_code = 'eng'
    else:
        english_query = translate_text_offline(user_input)

    if not user_input.strip():
        return "", None, history, default_map, None

    # ---> HIGHLIGHT AND REPLACE YOUR OLD FAQ_LINK CODE WITH THIS <---
    import urllib.parse
    faq_link = ""
    location = CURRENT_LOCATION_NAME if 'CURRENT_LOCATION_NAME' in globals() else "this location"
    
    if 'DIRECT_FAQ_URL' in globals() and DIRECT_FAQ_URL:
        faq_link = f"\n\n[🔗 Click here for the official {location} FAQ]({DIRECT_FAQ_URL})"
    else:
        encoded_loc = urllib.parse.quote(location + " FAQ")
        faq_link = f"\n\n[🔍 Click here to search for the {location} FAQ](https://www.google.com/search?q={encoded_loc})"
    # ----------------------------------------------------------------

    intent = route_question(english_query)


    # ---------------------------------------------------------
    # ROUTE 1: CHAT
    # ---------------------------------------------------------
    if "CHAT" in intent:
        # We just read the variable directly, no 'global' keyword needed!
        location_context = CURRENT_LOCATION_NAME if 'CURRENT_LOCATION_NAME' in globals() else "this location"

        guardrail_prompt = f"""
        You are an expert Local Tourist and Spatial Guide specifically for {location_context}.

        YOUR ALLOWED TOPICS:
        - You may use your general knowledge to answer questions about the history, culture, and facts regarding {location_context}.
        - You may help users understand locations.

        YOUR STRICT RESTRICTIONS (GUARDRAILS):
        - If the user asks about programming, coding, mathematics, or science, YOU MUST REFUSE.
        - If the user asks for recipes, medical advice, or political opinions, YOU MUST REFUSE.
        - If the user asks a completely random question about a DIFFERENT location, YOU MUST REFUSE.

        If a question violates these rules, apologize and politely say: "I am a spatial map guide. I can only help you with geographic questions or local history regarding {location_context}!"
        """

        qwen_english_response = get_chat_response(user_input, guardrail_prompt)

        final_text_response = translate_english_to_target(qwen_english_response, input_lang_code)
        final_audio_text = translate_english_to_target(qwen_english_response, audio_lang_code)

        history.append({"role": "user", "content": user_input})
        history.append({"role": "assistant", "content": final_text_response + faq_link})

        return "", None, history, default_map, generate_audio_response(final_audio_text, audio_lang_code)


    # ---------------------------------------------------------
    # ROUTE 2: RAG POLICY (3-Layer Fallback)
    # ---------------------------------------------------------
    elif "POLICY" in intent:
        # LAYER 1: FAISS RAG (Wikipedia + uploaded PDF)
        qwen_english_response = generate_rag_response(english_query, "eng")

        # LAYER 2: If RAG failed, try Universal Handbook
        if "NOT_FOUND" in qwen_english_response or "upload" in qwen_english_response.lower():

            domain_type = domain_choice if 'domain_choice' in dir() else "Tourist Area"
            handbook = DOMAIN_HANDBOOKS.get(domain_type, "")

            if handbook:
                handbook_prompt = f"""You are an expert local guide. Use the following universal rules to answer the user's question.
                RULES AND GUIDELINES:
                {handbook}

                Answer the user's question strictly based on these rules. If the question is not covered by these rules, you MUST output exactly this phrase: 'NOT_FOUND_IN_HANDBOOK'
                """
                handbook_response = get_chat_response(english_query, handbook_prompt)

                # Check for the correct phrase that we just told the handbook to output!
                if "NOT_FOUND_IN_HANDBOOK" not in handbook_response:
                    qwen_english_response = handbook_response
                else:
                    # LAYER 3: Qwen general knowledge fallback
                    location_context = CURRENT_LOCATION_NAME if 'CURRENT_LOCATION_NAME' in globals() else "this location"

                    guardrail_prompt = f"""
                    You are an expert Local Tourist and Spatial Guide specifically for {location_context}.

                    YOUR ALLOWED TOPICS:
                    - You may use your general knowledge to answer ANY general questions regarding traveling in college campuses, tourist locations, and temples.
                    - You may answer questions about the history, culture, trivia, tourism logistics, and general facts regarding {location_context}.
                    - You may give advice on how to navigate the area, what to wear, weather, transportation, or best times to visit.

                    YOUR STRICT RESTRICTIONS (GUARDRAILS):
                    - If the user asks to write code, solve math, or answer science questions, YOU MUST REFUSE.
                    - If the user asks for medical advice, recipes, or political opinions, YOU MUST REFUSE.
                    - If the user asks a completely random question about an unrelated topic, YOU MUST REFUSE.

                    If a question violates these rules, apologize and politely say: "I am a spatial map guide. I can only help you with geographic questions, travel advice, or local information regarding {location_context}!"
                    """

                    chat_fallback_response = get_chat_response(english_query, guardrail_prompt)
                    qwen_english_response = f"I couldn't find the exact rule in the handbook or Wikipedia, but based on my general knowledge: \n\n{chat_fallback_response}"

        final_text_response = translate_english_to_target(qwen_english_response, input_lang_code)
        final_audio_text = translate_english_to_target(qwen_english_response, audio_lang_code)

        history.append({"role": "user", "content": user_input})
        history.append({"role": "assistant", "content": final_text_response + faq_link})

        return "", None, history, default_map, generate_audio_response(final_audio_text, audio_lang_code)


    # ---------------------------------------------------------
    # ROUTE 2.5: CLARIFICATION (Missing Anchor)
    # ---------------------------------------------------------
    elif "CLARIFICATION" in intent:
        qwen_english_response = "Nearest to where? Please tell me which building you are near, or just ask me where all the locations are for a general list."

        final_text_response = translate_english_to_target(qwen_english_response, input_lang_code)
        final_audio_text = translate_english_to_target(qwen_english_response, audio_lang_code)

        history.append({"role": "user", "content": user_input})
        history.append({"role": "assistant", "content": final_text_response})

        return "", None, history, default_map, generate_audio_response(final_audio_text, audio_lang_code)


    # ---------------------------------------------------------
    # ROUTE 3: DB MAP
    # ---------------------------------------------------------
    max_retries, attempt, gdf, sql_query, feedback_context = 1, 1, None, "", ""
    while attempt <= max_retries + 1:
        sql_query = get_sql(english_query, feedback_context=feedback_context)
        gdf = get_db_results(sql_query)
        if gdf is not None and len(gdf) > 0: break
        feedback_context = f"Your previous query:\n```sql\n{sql_query}\n```\nreturned exactly 0 rows. Try using broader synonyms and avoid restrictive AND conditions."
        attempt += 1

    if gdf is None or len(gdf) == 0:
        qwen_english_error = get_friendly_response(user_input, "The database search returned 0 results. You MUST apologize to the user and say you cannot find it on the map. DO NOT try to answer the user's question using your general knowledge.")

        final_text_error = translate_english_to_target(qwen_english_error, input_lang_code)
        final_audio_error_text = translate_english_to_target(qwen_english_error, audio_lang_code)


        osm_prompt = (
            "\n\n> 📍 **Location Not Found!**\n"
            "> We couldn't find any places matching your request in our database. "
            "> Since our spatial engine is powered by community mapping, this likely means the location hasn't been mapped yet! "
            "> You can help improve the map by adding it directly on <a href='https://www.openstreetmap.org/edit' target='_blank'>OpenStreetMap.org</a>."
        )

        error_msg = final_text_error + osm_prompt

        history.append({"role": "user", "content": user_input})
        history.append({"role": "assistant", "content": error_msg + faq_link})

        return "", None, history, default_map, generate_audio_response(final_audio_error_text, audio_lang_code)


    # Standard Success Route
    qwen_english_success = get_friendly_response(user_input, gdf)

    final_text_success = translate_english_to_target(qwen_english_success, input_lang_code)
    final_audio_success_text = translate_english_to_target(qwen_english_success, audio_lang_code)

    success_msg = final_text_success

    # RESTORED TO DICTIONARY FORMAT
    history.append({"role": "user", "content": user_input})
    history.append({"role": "assistant", "content": success_msg + faq_link})

    return "", None, history, generate_map_html(gdf), generate_audio_response(final_audio_success_text, audio_lang_code)



if __name__ == "__main__":
    with gr.Blocks(theme=gr.themes.Soft(), title="Universal AI Guide", css=CUSTOM_CSS) as demo:
        gr.HTML("<div class='hero-section'><h1>🌍 Universal AI Local Guide</h1><p>Powered by SeamlessM4T & Qwen2.5-Coder!</p></div>")

        with gr.Row():
            college_input = gr.Textbox(label="Step 1: Enter College, Temple, or Tourist Landmark", placeholder="e.g., IIT Delhi, Taj Mahal, Tirupati Balaji...", scale=3)
            btn_load = gr.Button("🌍 Map & Load Database", elem_classes="custom-button", size="lg", scale=1)
            btn_audit = gr.Button("📊 Audit Map Data", elem_classes="custom-button", size="lg", scale=1)

        with gr.Row():
            domain_dropdown = gr.Dropdown(
                choices=["College Campus", "Temple / Monument", "Tourist Area"],
                value="Tourist Area",
                label="Location Type (For Offline Rulebook)",
                scale=1
            )
            pdf_upload = gr.File(label="Optional: Upload Rulebook or Tourist Guidebook (PDF) for Q&A", file_types=[".pdf"], scale=2)


        status_box = gr.Textbox(label="System Status", interactive=False, value="✨ Ready! Map a location first.", elem_classes="status-box")

        btn_load.click(fn=load_map_data, inputs=college_input, outputs=status_box)
        btn_audit.click(fn=ui_audit_data, outputs=status_box)
        pdf_upload.upload(fn=process_pdf, inputs=pdf_upload, outputs=status_box)

        with gr.Group(elem_classes="chat-container"):
            with gr.Row():
                with gr.Column(scale=4):
                    gr.HTML("<div class='chat-header'>💬 Chat with your Local Guide</div>")

                    # CHANGED: REMOVED type="messages"
                    chatbot = gr.Chatbot(height=500, elem_classes="gradio-chatbot")

                    with gr.Row():
                        msg_input = gr.Textbox(placeholder="✨ Type or speak your message...", container=False, scale=3)
                        mic_input = gr.Audio(sources=["microphone"], type="filepath", scale=1, container=False)

                        tts_lang = gr.Dropdown(
                            choices=[
                                "English (eng)",
                                "Hindi (hin)",
                                "Bengali (ben)",
                                "Telugu (tel)",
                                "Urdu (urd)"
                            ],
                            value="English (eng)",
                            label="Response Language",
                            scale=1
                        )
                        send_btn = gr.Button("Send", scale=1)

                    audio_output = gr.Audio(visible=True, autoplay=True, label="Audio Response")

                with gr.Column(scale=6):
                    gr.HTML("<div class='chat-header'>📍 Interactive Campus Map</div>")
                    map_output = gr.HTML(value="<div style='height: 500px; display: flex; align-items: center; justify-content: center; background: #f8f9fa; border-radius: 12px; border: 1px solid #e0e0e0; color: #888;'><h4>Map will appear here after a spatial query</h4></div>")

        msg_input.submit(fn=chat_wrapper, inputs=[msg_input, mic_input, chatbot, tts_lang, domain_dropdown], outputs=[msg_input, mic_input, chatbot, map_output, audio_output])
        send_btn.click(fn=chat_wrapper, inputs=[msg_input, mic_input, chatbot, tts_lang, domain_dropdown], outputs=[msg_input, mic_input, chatbot, map_output, audio_output])

    demo.launch(share=True, debug=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://4b4eb5ada74802d019.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



🌍 Geocoding 'Birla Institute of Technology and Science - BITS'...
🌍 Fetching spatial data for Birla Institute of Technology and Science - BITS at (28.3586648, 75.5882558)...
💾 Writing 140 features to PostgreSQL (campus_places)...
📚 Wikipedia loaded! 91 chunks from 'BITS Pilani' added to RAG.
✅ Found Direct FAQ Link: https://www.engineering.iastate.edu/international-collaborative-education-programs/bachelors-degree-program-2-2/bits-pilani-dual-degree/
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://4b4eb5ada74802d019.gradio.live
